# 04 — Feature Engineering

**Sources:** `feature_engineering.ipynb` (final FE section) + `df_knee_exploration.ipynb` (feature engineering sections)

Covers: comorbidity count, OKS subscales, OKS severity categories, age/BMI bins, interaction features, one-hot encoding.

# Final Feature Engineering

In [1]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

# Load preprocessed train and test data
X_train_final = pd.read_parquet('../data/cleaned/X_train_preprocessed.parquet')
X_test_final = pd.read_parquet('../data/cleaned/X_test_preprocessed.parquet')
y_train = pd.read_parquet('../data/cleaned/y_train.parquet').squeeze()
y_test = pd.read_parquet('../data/cleaned/y_test.parquet').squeeze()

print("="*70)
print("DATA LOADED SUCCESSFULLY")
print("="*70)
print(f"✓ X_train shape: {X_train_final.shape}")
print(f"✓ X_test shape: {X_test_final.shape}")
print(f"✓ y_train shape: {y_train.shape}")
print(f"✓ y_test shape: {y_test.shape}")

DATA LOADED SUCCESSFULLY
✓ X_train shape: (105905, 40)
✓ X_test shape: (26477, 40)
✓ y_train shape: (105905,)
✓ y_test shape: (26477,)


In [2]:
# Read data from proms_providers_academic_categories.xlsx

proms_academic_df = pd.read_excel('../data/supporting/proms_providers_academic_categories.xlsx')

proms_academic_df.head()

# Transform academisch/niet academisch to 1/0
proms_academic_df['University Hospital'] = proms_academic_df['category'].map({'academisch': 1, 'niet-academisch': 0})
proms_academic_df.head(50)

# Merge with X_train_final and X_test_final
X_train_final = X_train_final.merge(proms_academic_df[['provider_code', 'University Hospital']], on='provider_code', how='left')
X_test_final = X_test_final.merge(proms_academic_df[['provider_code', 'University Hospital']], on='provider_code', how='left')

X_train_final['University Hospital'].value_counts()
X_test_final['University Hospital'].value_counts()

University Hospital
0    20694
1     5783
Name: count, dtype: int64

In [3]:
# Read data from proms_providers_academic_categories.xlsx

proms_hospital_info = pd.read_excel('../data/supporting/ODS Advanced Search 2.xlsx')

# Example - will customize based on your actual values
proms_hospital_info['independent_hospital'] = proms_hospital_info['Primary Role Name'].map({
    'INDEPENDENT SECTOR H/C PROVIDER SITE': 1,
    'NHS TRUST': 0,
    'INDEPENDENT SECTOR HEALTHCARE PROVIDER': 1
})

proms_hospital_info['provider_code'] = proms_hospital_info['Dataset']
# Merge with X_train_final and X_test_final
X_train_final = X_train_final.merge(proms_hospital_info[['provider_code', 'independent_hospital', 'Region']], on='provider_code', how='left')
X_test_final = X_test_final.merge(proms_hospital_info[['provider_code', 'independent_hospital', 'Region']], on='provider_code', how='left')

X_train_final['Region'].value_counts()
proms_hospital_info['Primary Role Name'].value_counts()

Primary Role Name
INDEPENDENT SECTOR H/C PROVIDER SITE      156
NHS TRUST                                 136
INDEPENDENT SECTOR HEALTHCARE PROVIDER      2
Name: count, dtype: int64

In [4]:
# Extract top 10 most common provider codes with their value counts and how much they appear in the training set (% wise of total)
top_50_providers = X_train_final['provider_code'].value_counts().nsmallest(50).index.tolist()

for provider in top_50_providers:
    count = X_train_final['provider_code'].value_counts()[provider]
    pct = (count / len(X_train_final)) * 100
    print(f"Provider Code: {provider}, Count: {count}, Percentage of Training Set: {pct:.2f}%")
# ============================================================

Provider Code: RJ7, Count: 5, Percentage of Training Set: 0.00%
Provider Code: NT30A, Count: 7, Percentage of Training Set: 0.01%
Provider Code: NYW01, Count: 8, Percentage of Training Set: 0.01%
Provider Code: NT310, Count: 9, Percentage of Training Set: 0.01%
Provider Code: NT343, Count: 9, Percentage of Training Set: 0.01%
Provider Code: NT455, Count: 10, Percentage of Training Set: 0.01%
Provider Code: NFH01, Count: 11, Percentage of Training Set: 0.01%
Provider Code: NT315, Count: 12, Percentage of Training Set: 0.01%
Provider Code: NT416, Count: 16, Percentage of Training Set: 0.02%
Provider Code: NVC0R, Count: 16, Percentage of Training Set: 0.02%
Provider Code: NQM01, Count: 17, Percentage of Training Set: 0.02%
Provider Code: NT308, Count: 20, Percentage of Training Set: 0.02%
Provider Code: NT241, Count: 22, Percentage of Training Set: 0.02%
Provider Code: NT436, Count: 22, Percentage of Training Set: 0.02%
Provider Code: NT211, Count: 23, Percentage of Training Set: 0.02%
Pr

### Comorbidity counter

In [5]:
# Create a new variable, comorbidity counter based on all conditions

health_conditions = [
    'heart_disease', 'high_bp', 'stroke', 'circulation',
    'lung_disease', 'diabetes', 'kidney_disease',
    'nervous_system', 'liver_disease', 'cancer',
    'depression', 'arthritis'
]
X_train_final['comorbidity_count'] = X_train_final[health_conditions].sum(axis=1)
X_test_final['comorbidity_count'] = X_test_final[health_conditions].sum(axis=1)

X_train_final['comorbidity_count'].value_counts().sort_index()

comorbidity_count
0     10920
1     36181
2     34655
3     16688
4      5460
5      1453
6       382
7        68
8        18
9         7
10       14
11       18
12       41
Name: count, dtype: int64

In [6]:
X_train_final.columns.tolist()

['provider_code',
 'year',
 'age_band',
 'gender',
 't0_assisted',
 't0_assisted_by',
 't0_symptom_period',
 't0_previous_surgery',
 't0_living_arrangements',
 't0_disability',
 'heart_disease',
 'high_bp',
 'stroke',
 'circulation',
 'lung_disease',
 'diabetes',
 'kidney_disease',
 'nervous_system',
 'liver_disease',
 'cancer',
 'depression',
 'arthritis',
 't0_mobility',
 't0_self_care',
 't0_activity',
 't0_discomfort',
 't0_anxiety',
 'oks_t0_pain',
 'oks_t0_night_pain',
 'oks_t0_washing',
 'oks_t0_transport',
 'oks_t0_walking',
 'oks_t0_standing',
 'oks_t0_limping',
 'oks_t0_kneeling',
 'oks_t0_work',
 'oks_t0_confidence',
 'oks_t0_shopping',
 'oks_t0_stairs',
 'oks_t0_score',
 'University Hospital',
 'independent_hospital',
 'Region',
 'comorbidity_count']

### OKS Subscales

In [8]:
# Create new variables, oks subscales for pain, function and activities of daily life

X_train_final['oks_pain_subscale'] = (
    X_train_final['oks_t0_pain'] + X_train_final['oks_t0_night_pain']
)
X_train_final['oks_function_subscale'] = (
    X_train_final['oks_t0_washing']  + X_train_final['oks_t0_stairs'] +
    X_train_final['oks_t0_walking'] + X_train_final['oks_t0_standing'] +
    X_train_final['oks_t0_limping'] + X_train_final['oks_t0_kneeling']
)
X_train_final['oks_adl_subscale'] = (
    X_train_final['oks_t0_work'] + X_train_final['oks_t0_confidence'] +
    X_train_final['oks_t0_shopping'] + X_train_final['oks_t0_transport']
)

# Apply same for test set

X_test_final['oks_pain_subscale'] = (
    X_test_final['oks_t0_pain'] + X_test_final['oks_t0_night_pain']
)

X_test_final['oks_function_subscale'] = (
    X_test_final['oks_t0_washing']  + X_test_final['oks_t0_stairs'] +
    X_test_final['oks_t0_walking'] + X_test_final['oks_t0_standing'] +
    X_test_final['oks_t0_limping'] + X_test_final['oks_t0_kneeling']
)

X_test_final['oks_adl_subscale'] = (
    X_test_final['oks_t0_work'] + X_test_final['oks_t0_confidence'] +
    X_test_final['oks_t0_shopping'] + X_test_final['oks_t0_transport']
)
X_train_final[['oks_pain_subscale', 'oks_function_subscale', 'oks_adl_subscale']].head()

,oks_pain_subscale,oks_function_subscale,oks_adl_subscale
0,2,10,8
1,0,12,11
2,0,8,5
3,2,10,10
4,3,14,10


In [9]:
# ============================================================
# ONE-HOT ENCODE CATEGORICAL VARIABLES
# ============================================================

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import joblib

# Define categorical columns to encode
categorical_columns = [
    'year', 'age_band', 
    't0_living_arrangements', 'Region'
]

print("="*70)
print("ONE-HOT ENCODING CATEGORICAL VARIABLES")
print("="*70)

# Check which categorical columns exist in the data
existing_categorical = [col for col in categorical_columns if col in X_train_final.columns]
print(f"\nCategorical columns found: {existing_categorical}")

# ============================================================
# FIT ONE-HOT ENCODER ON TRAINING DATA
# ============================================================

# Initialize encoder
encoder = OneHotEncoder(
    drop='first',           # Drop first category to avoid multicollinearity
    sparse_output=False,    # Return dense array (easier to work with)
    handle_unknown='ignore' # Ignore unknown categories in test set
)

# Fit on training data
print(f"\n✓ Fitting encoder on training data...")
encoder.fit(X_train_final[existing_categorical])

# Get feature names for encoded columns
encoded_feature_names = encoder.get_feature_names_out(existing_categorical)
print(f"✓ Encoder fitted - will create {len(encoded_feature_names)} one-hot encoded features")

# ============================================================
# TRANSFORM TRAINING DATA
# ============================================================

print(f"\n✓ Transforming training data...")

# Transform categorical columns
X_train_encoded = encoder.transform(X_train_final[existing_categorical])

# Create DataFrame with encoded features
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train_final.index
)

# Drop original categorical columns and add encoded ones
X_train_numerical = X_train_final.drop(columns=existing_categorical)
X_train_final_encoded = pd.concat([X_train_numerical, X_train_encoded_df], axis=1)

print(f"✓ Training data shape before encoding: {X_train_final.shape}")
print(f"✓ Training data shape after encoding: {X_train_final_encoded.shape}")

# ============================================================
# TRANSFORM TEST DATA
# ============================================================

print(f"\n✓ Transforming test data...")

# Transform categorical columns (using fitted encoder)
X_test_encoded = encoder.transform(X_test_final[existing_categorical])

# Create DataFrame with encoded features
X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test_final.index
)

# Drop original categorical columns and add encoded ones
X_test_numerical = X_test_final.drop(columns=existing_categorical)
X_test_final_encoded = pd.concat([X_test_numerical, X_test_encoded_df], axis=1)

print(f"✓ Test data shape before encoding: {X_test_final.shape}")
print(f"✓ Test data shape after encoding: {X_test_final_encoded.shape}")

# ============================================================
# VERIFICATION
# ============================================================

print("\n" + "="*70)
print("ENCODING VERIFICATION")
print("="*70)

print(f"\n✓ X_train_final_encoded shape: {X_train_final_encoded.shape}")
print(f"✓ X_test_final_encoded shape: {X_test_final_encoded.shape}")
print(f"✓ Both datasets have same number of features: {X_train_final_encoded.shape[1] == X_test_final_encoded.shape[1]}")

# Check for missing values
print(f"\n✓ Missing values in X_train_final_encoded: {X_train_final_encoded.isna().sum().sum()}")
print(f"✓ Missing values in X_test_final_encoded: {X_test_final_encoded.isna().sum().sum()}")

# Show sample of new columns
print("\n" + "="*70)
print("SAMPLE OF ONE-HOT ENCODED FEATURES")
print("="*70)

# Show first few encoded feature names for each original categorical variable
for original_col in existing_categorical:
    encoded_cols = [col for col in encoded_feature_names if col.startswith(original_col)]
    print(f"\n{original_col} → {len(encoded_cols)} one-hot features:")
    print(f"  Sample: {encoded_cols[:5]}")

# Show data sample
print("\n" + "="*70)
print("DATA PREVIEW (First 5 rows, first 10 columns)")
print("="*70)
print(X_train_final_encoded.iloc[:5, :10])

# ============================================================
# SAVE ENCODED DATA (Optional)
# ============================================================

print("\n" + "="*70)
print("SAVING ENCODED DATA")
print("="*70)

# Drop provider_code before saving

X_train_final_encoded = X_train_final_encoded.drop(columns=['provider_code'])
X_test_final_encoded = X_test_final_encoded.drop(columns=['provider_code'])

# Save encoded datasets
X_train_final_encoded.to_parquet('../data/cleaned/X_train_encoded_wo_provider.parquet', index=False)
X_test_final_encoded.to_parquet('../data/cleaned/X_test_encoded_wo_provider.parquet', index=False)

# Save encoder for future use
joblib.dump(encoder, '../data/cleaned/onehot_encoder_wo_provider.pkl')

print("✓ X_train_final_encoded saved to '../data/cleaned/X_train_encoded.parquet'")
print("✓ X_test_final_encoded saved to '../data/cleaned/X_test_encoded.parquet'")
print("✓ OneHotEncoder saved to '../data/cleaned/onehot_encoder.pkl'")

# ============================================================
# FEATURE COUNTS BY TYPE
# ============================================================

print("\n" + "="*70)
print("FEATURE BREAKDOWN")
print("="*70)

# Count features by original type
numerical_features = X_train_numerical.columns.tolist()
encoded_features = encoded_feature_names.tolist()

print(f"\nNumerical features (unchanged): {len(numerical_features)}")
print(f"One-hot encoded features (new): {len(encoded_features)}")
print(f"Total features: {len(X_train_final_encoded.columns)}")

print("\n" + "="*70)
print("✓ ONE-HOT ENCODING COMPLETE - DATA READY FOR MODELING")
print("="*70)

# ============================================================
# How to load encoder and apply to new data later:
# ============================================================
"""
# Load the fitted encoder
encoder = joblib.load('../data/processed/onehot_encoder.pkl')

# Apply to new data
X_new_encoded = encoder.transform(X_new[existing_categorical])
X_new_encoded_df = pd.DataFrame(
    X_new_encoded,
    columns=encoder.get_feature_names_out(existing_categorical),
    index=X_new.index
)
X_new_numerical = X_new.drop(columns=existing_categorical)
X_new_final = pd.concat([X_new_numerical, X_new_encoded_df], axis=1)
"""

ONE-HOT ENCODING CATEGORICAL VARIABLES

Categorical columns found: ['year', 'age_band', 't0_living_arrangements', 'Region']

✓ Fitting encoder on training data...
✓ Encoder fitted - will create 15 one-hot encoded features

✓ Transforming training data...
✓ Training data shape before encoding: (105905, 47)
✓ Training data shape after encoding: (105905, 58)

✓ Transforming test data...
✓ Test data shape before encoding: (26477, 47)
✓ Test data shape after encoding: (26477, 58)

ENCODING VERIFICATION

✓ X_train_final_encoded shape: (105905, 58)
✓ X_test_final_encoded shape: (26477, 58)
✓ Both datasets have same number of features: True

✓ Missing values in X_train_final_encoded: 0
✓ Missing values in X_test_final_encoded: 0

SAMPLE OF ONE-HOT ENCODED FEATURES

year → 2 one-hot features:
  Sample: ['year_2017/18', 'year_2018/19']

age_band → 3 one-hot features:
  Sample: ['age_band_3', 'age_band_4', 'age_band_5']

t0_living_arrangements → 2 one-hot features:
  Sample: ['t0_living_arrangeme

"\n# Load the fitted encoder\nencoder = joblib.load('../data/processed/onehot_encoder.pkl')\n\n# Apply to new data\nX_new_encoded = encoder.transform(X_new[existing_categorical])\nX_new_encoded_df = pd.DataFrame(\n    X_new_encoded,\n    columns=encoder.get_feature_names_out(existing_categorical),\n    index=X_new.index\n)\nX_new_numerical = X_new.drop(columns=existing_categorical)\nX_new_final = pd.concat([X_new_numerical, X_new_encoded_df], axis=1)\n"

---
# --- df_knee_exploration.ipynb: Feature Engineering sections ---
The cells below come from `df_knee_exploration.ipynb` (sections 7–12: engineered features, summary, data quality check).

### 7.1 Baseline OKS Categories

In [23]:
# Create OKS severity categories based on baseline
if 'oks_t0_score' in df_engineered.columns:
    # OKS scale: 0-48, higher is better
    # Categories based on knee function severity
    def categorize_oks(score):
        if pd.isna(score):
            return 'Unknown'
        elif score < 20:
            return 'Severe'  # Very poor function
        elif score < 30:
            return 'Moderate'  # Moderate impairment
        elif score < 40:
            return 'Mild'  # Mild impairment
        else:
            return 'Minimal'  # Good function
    
    df_engineered['oks_t0_category'] = df_engineered['oks_t0_score'].apply(categorize_oks)
    
    # Binary: severe baseline (potential ceiling effect for improvement)
    df_engineered['oks_t0_severe'] = (df_engineered['oks_t0_score'] < 20).astype(int)
    
    # Room for improvement (higher baseline = less room to improve)
    df_engineered['oks_room_for_improvement'] = 48 - df_engineered['oks_t0_score']
    
    print("✓ Created OKS baseline features")
    print("\nOKS t0 category distribution:")
    print(df_engineered['oks_t0_category'].value_counts())
    
    print("\nGood outcome rate by OKS category:")
    print(df_engineered.groupby('oks_t0_category')['target'].mean().sort_values(ascending=False))

✓ Created OKS baseline features

OKS t0 category distribution:
oks_t0_category
Severe      70791
Moderate    48670
Mild        12257
Minimal       664
Name: count, dtype: int64

Good outcome rate by OKS category:
oks_t0_category
Minimal     0.954819
Mild        0.373664
Moderate    0.171564
Severe      0.111723
Name: target, dtype: float64


### 7.2 Age-Related Features

In [24]:
# Age-based features
age_cols = [col for col in df_engineered.columns if 'age' in col.lower()]

if age_cols:
    age_col = age_cols[0]
    
    # Age groups
    def categorize_age(age):
        if pd.isna(age):
            return 'Unknown'
        elif age < 50:
            return '<50'
        elif age < 65:
            return '50-64'
        elif age < 75:
            return '65-74'
        else:
            return '75+'
    
    df_engineered['age_group'] = df_engineered[age_col].apply(categorize_age)
    
    # Elderly flag (higher risk for complications)
    df_engineered['is_elderly'] = (df_engineered[age_col] >= 75).astype(int)
    
    # Young patient (different expectations/outcomes)
    df_engineered['is_young'] = (df_engineered[age_col] < 50).astype(int)
    
    print("✓ Created age-related features")
    print("\nAge group distribution:")
    print(df_engineered['age_group'].value_counts())
    
    print("\nGood outcome rate by age group:")
    print(df_engineered.groupby('age_group')['target'].mean().sort_values(ascending=False))

TypeError: '<' not supported between instances of 'str' and 'int'

### 7.3 BMI Categories

In [ ]:
# BMI-related features
bmi_cols = [col for col in df_engineered.columns if 'bmi' in col.lower() and 't1' not in col.lower()]

if bmi_cols:
    bmi_col = bmi_cols[0]
    
    # WHO BMI categories
    def categorize_bmi(bmi):
        if pd.isna(bmi):
            return 'Unknown'
        elif bmi < 18.5:
            return 'Underweight'
        elif bmi < 25:
            return 'Normal'
        elif bmi < 30:
            return 'Overweight'
        elif bmi < 35:
            return 'Obese_Class1'
        elif bmi < 40:
            return 'Obese_Class2'
        else:
            return 'Obese_Class3'
    
    df_engineered['bmi_category'] = df_engineered[bmi_col].apply(categorize_bmi)
    
    # Obesity flags (known risk factor)
    df_engineered['is_obese'] = (df_engineered[bmi_col] >= 30).astype(int)
    df_engineered['is_severely_obese'] = (df_engineered[bmi_col] >= 35).astype(int)
    
    print("✓ Created BMI-related features")
    print("\nBMI category distribution:")
    print(df_engineered['bmi_category'].value_counts())
    
    print("\nGood outcome rate by BMI category:")
    print(df_engineered.groupby('bmi_category')['target'].mean().sort_values(ascending=False))

### 7.4 Comorbidity Count

In [ ]:
# Create comorbidity count (if comorbidity columns exist)
# Look for binary comorbidity indicators
potential_comorbidity_cols = [col for col in df_engineered.columns if 
    any(x in col.lower() for x in ['diabetes', 'heart', 'hypertension', 'copd', 'asthma', 
                                     'kidney', 'liver', 'cancer', 'depression', 'anxiety']) 
    and 't0' in col.lower()]

if potential_comorbidity_cols:
    print(f"Found {len(potential_comorbidity_cols)} potential comorbidity columns")
    
    # Count comorbidities
    comorbidity_count = df_engineered[potential_comorbidity_cols].sum(axis=1)
    df_engineered['comorbidity_count'] = comorbidity_count
    
    # High comorbidity burden
    df_engineered['high_comorbidity_burden'] = (comorbidity_count >= 3).astype(int)
    
    print("✓ Created comorbidity count features")
    print("\nComorbidity count distribution:")
    print(df_engineered['comorbidity_count'].value_counts().sort_index())
    
    print("\nGood outcome rate by comorbidity count:")
    print(df_engineered.groupby('comorbidity_count')['target'].mean())
else:
    print("No comorbidity columns found for counting")

### 7.5 Interaction Features

In [ ]:
# Create meaningful interactions

# Age x OKS interaction (older patients with worse baseline)
if 'oks_t0_score' in df_engineered.columns and age_cols:
    age_col = age_cols[0]
    df_engineered['age_x_oks_t0'] = df_engineered[age_col] * df_engineered['oks_t0_score']
    print("✓ Created age x OKS interaction")

# Age x BMI interaction
if bmi_cols and age_cols:
    df_engineered['age_x_bmi'] = df_engineered[age_cols[0]] * df_engineered[bmi_cols[0]]
    print("✓ Created age x BMI interaction")

# Elderly with high BMI (compound risk)
if 'is_elderly' in df_engineered.columns and 'is_obese' in df_engineered.columns:
    df_engineered['elderly_obese'] = df_engineered['is_elderly'] * df_engineered['is_obese']
    print("✓ Created elderly-obese compound risk feature")

# Poor baseline with high comorbidity
if 'oks_t0_severe' in df_engineered.columns and 'comorbidity_count' in df_engineered.columns:
    df_engineered['severe_with_comorbidity'] = df_engineered['oks_t0_severe'] * df_engineered['comorbidity_count']
    print("✓ Created severe OKS with comorbidity interaction")

### 7.6 Surgery-Related Features

In [ ]:
# Analyze surgery-related features
surgery_cols = [col for col in df_engineered.columns if any(x in col.lower() for x in 
    ['procedure', 'surgery', 'operation', 'revision', 'bilateral', 'unilateral', 'side'])]

print(f"Surgery-related columns: {len(surgery_cols)}")
for col in surgery_cols:
    print(f"  - {col}")
    if df_engineered[col].dtype in ['object', 'category']:
        print(f"    Values: {df_engineered[col].value_counts().to_dict()}")

### 7.7 Provider/Hospital Volume Features

In [ ]:
# Provider volume (if available)
provider_cols = [col for col in df_engineered.columns if any(x in col.lower() for x in 
    ['provider', 'hospital', 'surgeon', 'volume'])]

print(f"Provider-related columns: {len(provider_cols)}")
for col in provider_cols[:5]:  # Show first 5
    print(f"  - {col}")
    print(f"    Unique values: {df_engineered[col].nunique()}")

## 8. Summary of Engineered Features

In [ ]:
# Identify new features
original_cols = set(df.columns)
new_cols = [col for col in df_engineered.columns if col not in original_cols]

print("="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"\nOriginal features: {len(original_cols)}")
print(f"New features created: {len(new_cols)}")
print(f"Total features: {len(df_engineered.columns)}")

print("\nNew Features:")
for i, col in enumerate(new_cols, 1):
    print(f"  {i:2d}. {col}")
    print(f"      Type: {df_engineered[col].dtype}")
    print(f"      Missing: {df_engineered[col].isnull().sum()} ({df_engineered[col].isnull().mean()*100:.1f}%)")
    if df_engineered[col].dtype in ['object', 'category']:
        print(f"      Unique values: {df_engineered[col].nunique()}")
    else:
        print(f"      Range: [{df_engineered[col].min():.2f}, {df_engineered[col].max():.2f}]")

## 9. Feature Importance Analysis (Univariate)

In [ ]:
# Analyze correlation of new features with target
new_numerical = [col for col in new_cols if df_engineered[col].dtype in [np.number, 'int64', 'float64']]

if new_numerical:
    new_corr = df_engineered[new_numerical + ['target']].corr()['target'].sort_values(ascending=False)
    new_corr = new_corr[new_corr.index != 'target']
    
    print("Correlation of New Features with Target:")
    print(new_corr)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['green' if x > 0 else 'red' for x in new_corr.values]
    ax.barh(range(len(new_corr)), new_corr.values, color=colors, alpha=0.7)
    ax.set_yticks(range(len(new_corr)))
    ax.set_yticklabels(new_corr.index)
    ax.set_xlabel('Correlation with Target')
    ax.set_title('New Features Correlation with Target', fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze categorical new features
new_categorical = [col for col in new_cols if df_engineered[col].dtype in ['object', 'category']]

for col in new_categorical:
    print(f"\n{'='*80}")
    print(f"Feature: {col}")
    print(f"{'='*80}")
    
    # Distribution
    print("\nDistribution:")
    print(df_engineered[col].value_counts())
    
    # Outcome rate by category
    print("\nGood outcome rate by category:")
    outcome_rate = df_engineered.groupby(col)['target'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    outcome_rate['mean'] = outcome_rate['mean'] * 100
    print(outcome_rate)
    
    # Chi-square test
    ct = pd.crosstab(df_engineered[col], df_engineered['target'])
    chi2, pval, dof, expected = stats.chi2_contingency(ct)
    print(f"\nChi-square test: χ²={chi2:.2f}, p-value={pval:.4f}")

## 10. Final Feature Set Preparation

In [ ]:
# Prepare final feature set (remove leakage features)
features_to_exclude = leakage_features + ['oks_improvement']  # Exclude all leakage features
features_to_exclude = [col for col in features_to_exclude if col in df_engineered.columns]

# Create final dataset
df_final = df_engineered.drop(columns=features_to_exclude)

print("="*80)
print("FINAL FEATURE SET")
print("="*80)
print(f"\nTotal features: {df_final.shape[1]}")
print(f"Total samples: {df_final.shape[0]:,}")
print(f"\nTarget distribution:")
print(df_final['target'].value_counts())
print(f"\nExcluded features (data leakage): {len(features_to_exclude)}")
for col in features_to_exclude:
    print(f"  - {col}")

In [ ]:
# Save engineered dataset
output_path = '../data/cleaned/knee-provider-featured.parquet'
df_final.to_parquet(output_path, index=False)
print(f"\n✓ Saved engineered dataset to: {output_path}")

## 11. Key Findings and Recommendations

In [ ]:
print("""
=====================================================================
KEY FINDINGS AND RECOMMENDATIONS FOR MODELING
=====================================================================

1. DATA LEAKAGE PREVENTION:
   ✓ Excluded all t1 (post-surgery) features
   ✓ Only using baseline (t0) and pre-surgery information
   ✓ Target variable uses t1 data ONLY for labeling

2. ENGINEERED FEATURES:
   - Baseline OKS categories (severity levels)
   - Room for improvement (ceiling effects)
   - Age groups and risk categories
   - BMI categories and obesity flags
   - Comorbidity burden counts
   - Interaction features (age x OKS, elderly + obese, etc.)

3. IMPORTANT PREDICTORS (based on correlation/analysis):
   - Baseline OKS score (oks_t0_score)
   - Age-related features
   - BMI and obesity status
   - Comorbidity burden
   - Surgery type/details

4. RECOMMENDATIONS FOR MODELING:
   a) Handle missing values appropriately:
      - Use imputation for critical features
      - Consider missing indicators for informative missingness
   
   b) Feature scaling:
      - Standardize numerical features
      - One-hot encode categorical features
   
   c) Class imbalance:
      - Check target distribution
      - Consider SMOTE, class weights, or stratified sampling
   
   d) Feature selection:
      - Use tree-based importance or LASSO
      - Consider domain knowledge
      - Remove highly correlated features
   
   e) Model selection:
      - Try multiple algorithms (LogReg, RF, XGBoost, LightGBM)
      - Use cross-validation
      - Focus on metrics relevant to healthcare (precision, recall, AUC)
   
   f) Validation strategy:
      - Use stratified k-fold CV
      - Consider temporal validation if date available
      - Hold out test set for final evaluation

5. NEXT STEPS:
   - Perform train-test split
   - Build preprocessing pipeline
   - Train baseline models
   - Tune hyperparameters
   - Evaluate and interpret results
""")

## 12. Data Quality Check Summary

In [ ]:
# Final data quality summary
print("="*80)
print("DATA QUALITY SUMMARY")
print("="*80)

# Missing values
missing_summary = df_final.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
print(f"\nFeatures with missing values: {len(missing_summary)}")
if len(missing_summary) > 0:
    print("\nTop 10 features by missing count:")
    print(missing_summary.head(10))

# Duplicates
duplicates = df_final.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Target distribution
print("\nTarget distribution:")
target_dist = df_final['target'].value_counts()
print(target_dist)
print(f"\nClass balance ratio: {target_dist.min() / target_dist.max():.2f}")

# Data types
print("\nData types distribution:")
print(df_final.dtypes.value_counts())

print("\n" + "="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)

In [ ]:
# Analyze variables for potential 9-based missing value indicators
# We'll check for patterns like 9, 99, 999, 9999 that might indicate missing values

def analyze_nine_patterns(df):
    """
    Analyze each column to detect if it uses 9-based patterns as missing indicators.
    Returns a DataFrame with columns that have suspicious 9-based patterns.
    """
    results = []
    
    for col in df.columns:
        col_data = df[col]
        
        # Skip non-numeric columns
        if not pd.api.types.is_numeric_dtype(col_data):
            continue
        
        # Get value counts
        value_counts = col_data.value_counts().sort_index()
        
        # Check for common missing value patterns
        patterns = {
            'has_9': 9 in value_counts.index,
            'has_99': 99 in value_counts.index,
            'has_999': 999 in value_counts.index,
            'has_9999': 9999 in value_counts.index,
            'count_9': value_counts.get(9, 0),
            'count_99': value_counts.get(99, 0),
            'count_999': value_counts.get(999, 0),
            'count_9999': value_counts.get(9999, 0),
        }
        
        # Calculate percentages
        total = len(col_data)
        patterns['pct_9'] = (patterns['count_9'] / total * 100) if patterns['count_9'] > 0 else 0
        patterns['pct_99'] = (patterns['count_99'] / total * 100) if patterns['count_99'] > 0 else 0
        patterns['pct_999'] = (patterns['count_999'] / total * 100) if patterns['count_999'] > 0 else 0
        patterns['pct_9999'] = (patterns['count_9999'] / total * 100) if patterns['count_9999'] > 0 else 0
        
        # Check if any 9-pattern exists
        has_any_pattern = any([patterns['has_9'], patterns['has_99'], 
                               patterns['has_999'], patterns['has_9999']])
        
        if has_any_pattern:
            # Get basic stats
            patterns['column'] = col
            patterns['dtype'] = str(col_data.dtype)
            patterns['unique_values'] = col_data.nunique()
            patterns['min'] = col_data.min()
            patterns['max'] = col_data.max()
            patterns['mean'] = col_data.mean()
            patterns['median'] = col_data.median()
            patterns['total_count'] = total
            patterns['actual_missing'] = col_data.isnull().sum()
            
            results.append(patterns)
    
    return pd.DataFrame(results)

print("Analyzing dataset for 9-based missing value indicators...")
print("="*80)

# Load the data
nine_patterns_df = analyze_nine_patterns(df)

if len(nine_patterns_df) > 0:
    # Sort by most frequent 9-pattern first
    nine_patterns_df['max_nine_count'] = nine_patterns_df[['count_9', 'count_99', 'count_999', 'count_9999']].max(axis=1)
    nine_patterns_df['max_nine_pct'] = nine_patterns_df[['pct_9', 'pct_99', 'pct_999', 'pct_9999']].max(axis=1)
    nine_patterns_df = nine_patterns_df.sort_values('max_nine_pct', ascending=False)
    
    print(f"\nFound {len(nine_patterns_df)} variables with 9-based patterns")
    print("\n" + "="*80)
else:
    print("\nNo 9-based patterns found in the dataset")

nine_patterns_df.head(20)

Analyzing dataset for 9-based missing value indicators...

Found 44 variables with 9-based patterns



,has_9,has_99,has_999,has_9999,count_9,count_99,count_999,count_9999,pct_9,pct_99,...,dtype,unique_values,min,max,mean,median,total_count,actual_missing,max_nine_count,max_nine_pct
13,True,False,False,False,129232,0,0,0,99.422232,0.000000,...,uint8,2,1.0,9.0,8.953779,9.0,129983,0,129232,99.422232
12,True,False,False,False,128694,0,0,0,99.008332,0.000000,...,uint8,2,1.0,9.0,8.920667,9.0,129983,0,128694,99.008332
7,True,False,False,False,127852,0,0,0,98.360555,0.000000,...,uint8,2,1.0,9.0,8.868844,9.0,129983,0,127852,98.360555
11,True,False,False,False,127361,0,0,0,97.982813,0.000000,...,uint8,2,1.0,9.0,8.838625,9.0,129983,0,127361,97.982813
14,True,False,False,False,122947,0,0,0,94.586984,0.000000,...,uint8,2,1.0,9.0,8.566959,9.0,129983,0,122947,94.586984
8,True,False,False,False,122817,0,0,0,94.486971,0.000000,...,uint8,2,1.0,9.0,8.558958,9.0,129983,0,122817,94.486971
23,True,False,False,False,119945,0,0,0,92.277452,0.000000,...,uint8,7,1.0,9.0,8.416062,9.0,129983,0,119945,92.277452
9,True,False,False,False,118139,0,0,0,90.888039,0.000000,...,uint8,2,1.0,9.0,8.271043,9.0,129983,0,118139,90.888039
15,True,False,False,False,117988,0,0,0,90.771870,0.000000,...,uint8,2,1.0,9.0,8.261750,9.0,129983,0,117988,90.771870
5,True,False,False,False,117940,0,0,0,90.734942,0.000000,...,uint8,2,1.0,9.0,8.258795,9.0,129983,0,117940,90.734942


In [ ]:
# Display detailed information about columns with 9-patterns
if len(nine_patterns_df) > 0:
    # Select key columns for display
    display_cols = ['column', 'has_9', 'count_9', 'pct_9', 
                    'has_99', 'count_99', 'pct_99',
                    'unique_values', 'min', 'max', 'median', 'actual_missing']
    
    print("\n" + "="*80)
    print("DETAILED ANALYSIS OF VARIABLES WITH 9-BASED PATTERNS")
    print("="*80)
    
    display_df = nine_patterns_df[display_cols].copy()
    display_df = display_df.round(2)
    
    # Format for better readability
    print("\nColumns with value = 9:")
    df_with_9 = display_df[display_df['has_9'] == True].sort_values('pct_9', ascending=False)
    if len(df_with_9) > 0:
        print(df_with_9[['column', 'count_9', 'pct_9', 'unique_values', 'min', 'max', 'median', 'actual_missing']].to_string())
    else:
        print("None found")
    
    print("\n" + "-"*80)
    print("Columns with value = 99:")
    df_with_99 = display_df[display_df['has_99'] == True].sort_values('pct_99', ascending=False)
    if len(df_with_99) > 0:
        print(df_with_99[['column', 'count_99', 'pct_99', 'unique_values', 'min', 'max', 'median', 'actual_missing']].to_string())
    else:
        print("None found")


DETAILED ANALYSIS OF VARIABLES WITH 9-BASED PATTERNS

Columns with value = 9:
                    column  count_9  pct_9  unique_values   min    max  median  actual_missing
13           liver_disease   129232  99.42              2   1.0    9.0     9.0               0
12          nervous_system   128694  99.01              2   1.0    9.0     9.0               0
7                   stroke   127852  98.36              2   1.0    9.0     9.0               0
11          kidney_disease   127361  97.98              2   1.0    9.0     9.0               0
14                  cancer   122947  94.59              2   1.0    9.0     9.0               0
8              circulation   122817  94.49              2   1.0    9.0     9.0               0
23          t1_assisted_by   119945  92.28              7   1.0    9.0     9.0               0
9             lung_disease   118139  90.89              2   1.0    9.0     9.0               0
15              depression   117988  90.77              2   1.0   